In [ ]:
# breakout_screener.ipynb

import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# Load data
def load_data(file_path):
    df = pd.read_csv(file_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    return df

# Resample data
def resample_data(df, freq):
    return df.set_index('date').groupby('symbol').resample(freq).agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()

# Horizontal resistance breakout
def detect_horizontal_resistance_breakout(df, lookback=20, buffer=0.5):
    breakout_stocks = []
    for symbol in df['symbol'].unique():
        data = df[df['symbol'] == symbol].copy()
        if len(data) < lookback + 1:
            continue
        recent = data.tail(lookback + 1)
        resistance = recent.head(lookback)['high'].max()
        last_close = recent.tail(1)['close'].values[0]
        if last_close > resistance + buffer:
            breakout_stocks.append({
                'symbol': symbol,
                'type': 'Horizontal Resistance Breakout',
                'date': recent.tail(1)['date'].values[0],
                'level': resistance,
                'close': last_close
            })
    return breakout_stocks

# Horizontal support reversal
def detect_horizontal_support_reversal(df, lookback=20, buffer=0.5):
    reversal_stocks = []
    for symbol in df['symbol'].unique():
        data = df[df['symbol'] == symbol].copy()
        if len(data) < lookback + 1:
            continue
        recent = data.tail(lookback + 1)
        support = recent.head(lookback)['low'].min()
        last_close = recent.tail(1)['close'].values[0]
        if last_close > support + buffer:
            reversal_stocks.append({
                'symbol': symbol,
                'type': 'Horizontal Support Reversal',
                'date': recent.tail(1)['date'].values[0],
                'level': support,
                'close': last_close
            })
    return reversal_stocks

# Trendline breakout/reversal
def detect_trendline(df, lookback=20, mode='resistance', buffer=0.5):
    trend_stocks = []
    for symbol in df['symbol'].unique():
        data = df[df['symbol'] == symbol].copy()
        if len(data) < lookback + 1:
            continue
        recent = data.tail(lookback + 1).copy()
        recent['x'] = np.arange(len(recent))
        if mode == 'resistance':
            y = recent['high'].values
        else:
            y = recent['low'].values
        X = recent['x'].values.reshape(-1, 1)
        model = LinearRegression().fit(X[:-1], y[:-1])
        trend_value = model.predict(X[-1].reshape(1, -1))[0]
        last_close = recent.tail(1)['close'].values[0]
        if last_close > trend_value + buffer:
            trend_stocks.append({
                'symbol': symbol,
                'type': f'Trendline {mode.capitalize()} Breakout' if mode == 'resistance' else f'Trendline {mode.capitalize()} Reversal',
                'date': recent.tail(1)['date'].values[0],
                'trend_value': trend_value,
                'close': last_close
            })
    return trend_stocks

# Run screener
def run_screener(df, label='daily'):
    results = []
    results += detect_horizontal_resistance_breakout(df)
    results += detect_horizontal_support_reversal(df)
    results += detect_trendline(df, mode='resistance')
    results += detect_trendline(df, mode='support')
    return pd.DataFrame(results)

# Main execution
def main():
    nse_path = 'bhav_data_nse.csv'
    bse_path = 'bhav_data_bse.csv'

    nse_daily = load_data(nse_path)
    bse_daily = load_data(bse_path)

    nse_weekly = resample_data(nse_daily, 'W-FRI')
    bse_weekly = resample_data(bse_daily, 'W-FRI')

    nse_monthly = resample_data(nse_daily, 'M')
    bse_monthly = resample_data(bse_daily, 'M')

    # Run screeners
    nse_opportunities = {
        'daily': run_screener(nse_daily, 'daily'),
        'weekly': run_screener(nse_weekly, 'weekly'),
        'monthly': run_screener(nse_monthly, 'monthly')
    }

    bse_opportunities = {
        'daily': run_screener(bse_daily, 'daily'),
        'weekly': run_screener(bse_weekly, 'weekly'),
        'monthly': run_screener(bse_monthly, 'monthly')
    }

    # Save results
    for freq in ['daily', 'weekly', 'monthly']:
        nse_opportunities[freq].to_csv(f'nse_{freq}_opportunities.csv', index=False)
        bse_opportunities[freq].to_csv(f'bse_{freq}_opportunities.csv', index=False)

    print("✅ Screener run complete. Results saved.")

main()
